In [2]:
import pandas as pd
import requests
import time
import random
import os
from datetime import datetime, timedelta
from tqdm import tqdm  # прогресс-бар (pip install tqdm)
from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout, RequestException

# 1. Парсинг файлов по сделкам

In [3]:
def download_tbank_trades_to_folder(
    instrument_id: str = "GAZP_TQBR",
    start_date: str = "2018-03-07",
    end_date: str = "2019-12-31",
    output_folder: str = "tbank_trades_GAZP",
    delay: float = 0.7
):
    """
    Скачивает обезличенные сделки (csv.gz) с T-Bank History MD в указанную папку.
    
    Параметры:
        instrument_id - например "GAZP_TQBR", "SBER_TQBR"
        start_date    - дата начала (включительно)
        end_date      - дата окончания (включительно)
        output_folder - папка, куда сохранять файлы
        delay         - пауза между запросами в секундах
    """
    
    # Создаём папку
    os.makedirs(output_folder, exist_ok=True)
    
    # Генерируем список дат (только торговые дни можно, но для простоты все дни)
    dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    print(f"🚀 Начинаем скачивание обезличенных сделок для {instrument_id}")
    print(f"   Период: {start_date} — {end_date}")
    print(f"   Папка: {output_folder}\n")

    downloaded = 0
    skipped = 0
    empty_files = 0

    for date_obj in tqdm(dates, desc="Обработка дней"):
        date_str = date_obj.strftime("%Y-%m-%d")
        
        url = f"https://invest-public-api.tinkoff.ru/history-trades/{date_str}?instrumentId={instrument_id}"
        filename = f"{instrument_id}_{date_str}.csv.gz"
        full_path = os.path.join(output_folder, filename)

        # Если файл уже существует и размер > 1 КБ — пропускаем
        if os.path.exists(full_path) and os.path.getsize(full_path) > 1024:
            skipped += 1
            continue

        try:
            response = requests.get(url, timeout=40)
            
            if response.status_code == 200:
                content = response.content
                
                # Проверка на пустой/почти пустой файл
                if len(content) < 500:   # очень маленький архив — скорее всего пустой день
                    empty_files += 1
                    print(f"⚠️  {date_str} — файл слишком маленький ({len(content)} байт), пропускаем")
                    continue
                
                with open(full_path, 'wb') as f:
                    f.write(content)
                
                downloaded += 1
                size_mb = len(content) / (1024 * 1024)
                print(f"✅ {date_str} — скачано {size_mb:.2f} МБ")
                
            elif response.status_code == 404:
                skipped += 1
                # Нет данных за этот день (выходной или нет сделок)
                continue
            else:
                print(f"❌ {date_str} — ошибка HTTP {response.status_code}")
                
        except Exception as e:
            print(f"❌ {date_str} — исключение: {e}")

        # Обязательная пауза, чтобы не получить блокировку
        time.sleep(delay + random.uniform(0.2, 0.6))

    print("\n🎉 Скачивание завершено!")
    print(f"   Скачано новых файлов : {downloaded}")
    print(f"   Пропущено (уже есть): {skipped}")
    print(f"   Пустых/маленьких     : {empty_files}")
    print(f"   Папка: {os.path.abspath(output_folder)}")






In [17]:
if __name__ == "__main__":
    import pandas as pd   # нужно для pd.date_range
    
    download_tbank_trades_to_folder(
        instrument_id="GAZP_TQBR",
        start_date="2023-01-01",
        end_date="2024-06-30",
        output_folder=r"C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr",
        delay=0.8
    )

🚀 Начинаем скачивание обезличенных сделок для GAZP_TQBR
   Период: 2023-01-01 — 2024-06-30
   Папка: C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr



Обработка дней:   0%|                                                                  | 1/547 [00:01<13:01,  1.43s/it]

❌ 2023-01-02 — исключение: ('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))


Обработка дней:   0%|▏                                                               | 2/547 [00:27<2:22:31, 15.69s/it]

✅ 2023-01-03 — скачано 0.21 МБ


Обработка дней:   1%|▎                                                               | 3/547 [00:31<1:33:42, 10.34s/it]

... (807 outputs left out)

Обработка дней: 100%|███████████████████████████████████████████████████████████████▊| 545/547 [58:49<00:18,  9.26s/it]

❌ 2024-06-29 — исключение: ('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))


Обработка дней: 100%|████████████████████████████████████████████████████████████████| 547/547 [59:12<00:00,  6.49s/it]


🎉 Скачивание завершено!
   Скачано новых файлов : 288
   Пропущено (уже есть): 140
   Пустых/маленьких     : 0
   Папка: C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr


# 2. Проверка полноты данных

In [4]:

def check_and_download_missing_days(
    instrument_id: str,
    start_date: str,
    end_date: str,
    output_folder: str,
    download_function=None,      # сюда передаём твою функцию скачивания
    min_file_size: int = 1024    # минимальный размер файла в байтах, чтобы считать его нормальным
):
    """
    Проверяет наличие файлов за каждый день в периоде и докачивает отсутствующие.
    
    Параметры:
        instrument_id   : например "GAZP_TQBR" или "SBER_TQBR"
        start_date      : '2020-01-01'
        end_date        : '2026-03-31'
        output_folder   : папка, где лежат файлы
        download_function : функция, которая скачивает данные за один день
                           (по умолчанию используем download_tbank_trades_to_folder)
        min_file_size   : минимальный размер файла (байты), ниже которого считаем файл некорректным
    """
    
#     if download_function is None:
#         from your_module import download_tbank_trades_to_folder as download_function   # замени на реальный импорт
    
    os.makedirs(output_folder, exist_ok=True)
    
    # Генерируем все календарные дни в периоде
    all_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    missing_dates = []
    bad_files = []          # файлы, которые существуют, но слишком маленькие (пустые/битые)

    print(f"🔍 Проверка файлов для {instrument_id} с {start_date} по {end_date}...\n")

    for date_obj in tqdm(all_dates, desc="Проверка дат"):
        date_str = date_obj.strftime('%Y-%m-%d')
        filename = f"{instrument_id}_{date_str}.csv.gz"          # ← твой формат
        full_path = os.path.join(output_folder, filename)
        print(full_path)

        if os.path.exists(full_path):
            size = os.path.getsize(full_path)
            if size < min_file_size:
                bad_files.append(date_str)
                missing_dates.append(date_str)   # считаем, что нужно перекачать
            # иначе — файл нормальный, пропускаем
        else:
            missing_dates.append(date_str)

    # Вывод результатов проверки
    print("\n" + "="*80)
    print(f"Результат проверки:")
    print(f"   Всего дней в периоде     : {len(all_dates)}")
    print(f"   Найдено корректных файлов: {len(all_dates) - len(missing_dates)}")
    print(f"   Отсутствует / битых      : {len(missing_dates)}")
    
    if bad_files:
        print(f"   Из них битых (маленьких): {len(bad_files)}")

    if not missing_dates:
        print("\n✅ Все файлы за период уже присутствуют и корректны!")
        return []

    print(f"\n📌 Найдено {len(missing_dates)} дней, которые нужно скачать:")
    for d in missing_dates[:10]:                     # показываем только первые 10
        print(f"   • {d}")
    if len(missing_dates) > 10:
        print(f"   ... и ещё {len(missing_dates)-10} дней")

    # === Запуск скачивания только отсутствующих дат ===
    print(f"\n🚀 Начинаем докачку недостающих дней...\n")

    for date_str in tqdm(missing_dates, desc="Докачка"):
        try:
            # Вызываем функцию скачивания **только за один день**
            download_function(
                instrument_id=instrument_id,
                start_date=date_str,
                end_date=date_str,           # скачиваем только один день
                output_folder=output_folder,
                delay=0.8
            )
            time.sleep(0.5)  # небольшая пауза между днями
        except Exception as e:
            print(f"❌ Ошибка при скачивании {date_str}: {e}")

    print("\n🎉 Проверка и докачка завершены!")
    return missing_dates



In [23]:
if __name__ == "__main__":
    missing = check_and_download_missing_days(
        instrument_id="GAZP_TQBR",
        start_date="2023-01-01",
        end_date="2026-03-31",           # или текущую дату
        output_folder=r"C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr",
        min_file_size=400,               # можно увеличить, если хочешь быть строже
        download_function = download_tbank_trades_to_folder
    )

🔍 Проверка файлов для GAZP_TQBR с 2023-01-01 по 2026-03-31...



Проверка дат:  81%|██████████████████████████████████████████████████▉            | 958/1186 [00:00<00:00, 4780.57it/s]

C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-01.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-02.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-03.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-04.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-05.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-06.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-07.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-08.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-09.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-10.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-11.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-12.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2023-01-13.csv.gz

Проверка дат: 100%|██████████████████████████████████████████████████████████████| 1186/1186 [00:00<00:00, 4320.53it/s]


C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-17.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-18.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-19.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-20.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-21.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-22.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-23.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-24.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-25.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-26.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-27.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-28.csv.gz
C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr\GAZP_TQBR_2025-08-29.csv.gz

Докачка:   0%|                                                                                 | 0/249 [00:00<?, ?it/s]

... (7 outputs left out)


Обработка дней: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:21<00:00, 21.76s/it]



🎉 Скачивание завершено!
   Скачано новых файлов : 0
   Пропущено (уже есть): 0
   Пустых/маленьких     : 0
   Папка: C:\Users\timur\Desktop\sd\tinvest_data\gazp\tqbr


Докачка:   0%|▎                                                                      | 1/249 [00:23<1:36:07, 23.26s/it]


KeyboardInterrupt: 

# 3. Парсинг файлов по годам

In [31]:

def download_tbank_all_trades_by_days(
    start_date: str,
    end_date: str = None,
    output_folder: str = "tbank_all_trades",
    delay: float = 0.75,
    min_file_size: int = 2000,      # байт — ниже этого считаем файл пустым/битым
    overwrite: bool = False         # True = перезаписывать существующие файлы
):
    """
    Скачивает обезличенные сделки ЗА ВСЕ инструменты по дням.
    Каждый день сохраняется в отдельный файл: YYYY-MM-DD.csv.gz
    
    Пример имени файла: 2026-04-06.csv.gz
    """
    if end_date is None:
        end_date = (datetime.today() - pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    os.makedirs(output_folder, exist_ok=True)

    # Генерируем все дни в периоде
    dates = pd.date_range(start=start_date, end=end_date, freq='D')

    print(f"🚀 Скачивание всех обезличенных сделок (по всем инструментам)")
    print(f"   Период: {start_date} — {end_date}")
    print(f"   Папка: {output_folder}\n")

    downloaded = 0
    skipped = 0
    small_files = 0

    for date_obj in tqdm(dates, desc="Скачивание дней"):
        date_str = date_obj.strftime('%Y-%m-%d')
        url = f"https://invest-public-api.tinkoff.ru/history-trades/{date_str}"
        
        filename = f"{date_str}.csv.gz"
        full_path = os.path.join(output_folder, filename)

        # Пропускаем, если файл уже есть и нормального размера
        if not overwrite and os.path.exists(full_path) and os.path.getsize(full_path) >= min_file_size:
            skipped += 1
            continue

        try:
            response = requests.get(url, timeout=60)

            if response.status_code == 200:
                content = response.content
                size_bytes = len(content)

                if size_bytes < min_file_size:
                    small_files += 1
                    print(f"⚠️  {date_str} — файл слишком маленький ({size_bytes} байт)")
                    continue

                with open(full_path, 'wb') as f:
                    f.write(content)

                size_mb = size_bytes / (1024 * 1024)
                downloaded += 1
                print(f"✅ {date_str} — скачано {size_mb:.2f} МБ")

            elif response.status_code == 404:
                skipped += 1
                # Нет данных за этот день (выходной)
                continue
            else:
                print(f"❌ {date_str} — HTTP {response.status_code}")

        except Exception as e:
            print(f"❌ {date_str} — ошибка: {e}")

        # Пауза, чтобы не получить 429 Too Many Requests
        time.sleep(delay + random.uniform(0.1, 0.4))

    print("\n" + "="*70)
    print("🎉 Скачивание завершено!")
    print(f"   Скачано новых файлов : {downloaded}")
    print(f"   Пропущено (уже есть) : {skipped}")
    print(f"   Слишком маленьких    : {small_files}")
    print(f"   Всего обработано дней: {len(dates)}")
    print(f"   Папка: {os.path.abspath(output_folder)}")




In [6]:
def check_and_download_missing_days_2(
    start_date: str,
    end_date: str,
    output_folder: str,
    download_function=None,      # сюда передаём твою функцию скачивания
    min_file_size: int = 1024    # минимальный размер файла в байтах, чтобы считать его нормальным
):
    """
    Проверяет наличие файлов за каждый день в периоде и докачивает отсутствующие.
    
    Параметры:
        start_date      : '2020-01-01'
        end_date        : '2026-03-31'
        output_folder   : папка, где лежат файлы
        download_function : функция, которая скачивает данные за один день
                           (по умолчанию используем download_tbank_trades_to_folder)
        min_file_size   : минимальный размер файла (байты), ниже которого считаем файл некорректным
    """
    
    if download_function is None:
        from your_module import download_tbank_trades_to_folder as download_function   # замени на реальный импорт
    
    os.makedirs(output_folder, exist_ok=True)
    
    # Генерируем все календарные дни в периоде
    all_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    missing_dates = []
    bad_files = []          # файлы, которые существуют, но слишком маленькие (пустые/битые)

    print(f"🔍 Проверка файлов с {start_date} по {end_date}...\n")

    for date_obj in tqdm(all_dates, desc="Проверка дат"):
        date_str = date_obj.strftime('%Y-%m-%d')
        filename = f"{date_str}.csv.gz"          # ← твой формат
        full_path = os.path.join(output_folder, filename)

        if os.path.exists(full_path):
            size = os.path.getsize(full_path)
            if size < min_file_size:
                bad_files.append(date_str)
                missing_dates.append(date_str)   # считаем, что нужно перекачать
            # иначе — файл нормальный, пропускаем
        else:
            missing_dates.append(date_str)

    # Вывод результатов проверки
    print("\n" + "="*80)
    print(f"Результат проверки:")
    print(f"   Всего дней в периоде     : {len(all_dates)}")
    print(f"   Найдено корректных файлов: {len(all_dates) - len(missing_dates)}")
    print(f"   Отсутствует / битых      : {len(missing_dates)}")
    
    if bad_files:
        print(f"   Из них битых (маленьких): {len(bad_files)}")

    if not missing_dates:
        print("\n✅ Все файлы за период уже присутствуют и корректны!")
        return []

    print(f"\n📌 Найдено {len(missing_dates)} дней, которые нужно скачать:")
    for d in missing_dates[:10]:                     # показываем только первые 10
        print(f"   • {d}")
    if len(missing_dates) > 10:
        print(f"   ... и ещё {len(missing_dates)-10} дней")

    # === Запуск скачивания только отсутствующих дат ===
    print(f"\n🚀 Начинаем докачку недостающих дней...\n")

    for date_str in tqdm(missing_dates, desc="Докачка"):
        try:
            # Вызываем функцию скачивания **только за один день**
            download_function(
                start_date=date_str,
                end_date=date_str,           # скачиваем только один день
                output_folder=output_folder,
                delay=0.8
            )
            time.sleep(0.5)  # небольшая пауза между днями
        except Exception as e:
            print(f"❌ Ошибка при скачивании {date_str}: {e}")

    print("\n🎉 Проверка и докачка завершены!")
    return missing_dates


In [7]:
# ====================== ПРИМЕР ИСПОЛЬЗОВАНИЯ ======================

if __name__ == "__main__":
    download_tbank_all_trades_by_days(
        start_date="2022-01-01",
        end_date="2022-12-31",           # или None — до вчера
        output_folder=r"D:\sd2\tinvest_data\2022",
        delay=0.8,
        overwrite=False                  # поставь True, если хочешь перезаписывать
    )

🚀 Скачивание всех обезличенных сделок (по всем инструментам)
   Период: 2022-01-01 — 2022-12-31
   Папка: D:\sd2\tinvest_data\2022



Скачивание дней:   0%|                                                                         | 0/365 [00:00<?, ?it/s]

⚠️  2022-01-01 — файл слишком маленький (84 байт)


Скачивание дней:   0%|▏                                                                | 1/365 [00:01<09:23,  1.55s/it]

⚠️  2022-01-02 — файл слишком маленький (120 байт)


Скачивание дней:   1%|▎                                                                | 2/365 [00:03<09:35,  1.58s/it]

... (599 outputs left out)

Скачивание дней:  83%|██████████████████████████████████████████████████▍          | 302/365 [2:46:53<43:43, 41.65s/it]

⚠️  2022-10-30 — файл слишком маленький (121 байт)


Скачивание дней:  83%|██████████████████████████████████████████████████▋          | 303/365 [2:47:06<34:11, 33.09s/it]


KeyboardInterrupt: 

In [12]:
## ликвидация пропусков по дням
if __name__ == "__main__":
    missing = check_and_download_missing_days_2(
        start_date="2022-01-01",
        end_date="2022-12-31",           # или текущую дату
        output_folder=r"D:\sd2\tinvest_data\2022",
        min_file_size=400,               # можно увеличить, если хочешь быть строже
        download_function = download_tbank_all_trades_by_days
    )

🔍 Проверка файлов с 2022-01-01 по 2022-12-31...



Проверка дат: 100%|████████████████████████████████████████████████████████████████| 365/365 [00:00<00:00, 6386.50it/s]



Результат проверки:
   Всего дней в периоде     : 365
   Найдено корректных файлов: 257
   Отсутствует / битых      : 108

📌 Найдено 108 дней, которые нужно скачать:
   • 2022-01-01
   • 2022-01-02
   • 2022-01-08
   • 2022-01-09
   • 2022-01-15
   • 2022-01-16
   • 2022-01-22
   • 2022-01-23
   • 2022-01-29
   • 2022-01-30
   ... и ещё 98 дней

🚀 Начинаем докачку недостающих дней...



Докачка:   0%|                                                                                 | 0/108 [00:00<?, ?it/s]

🚀 Скачивание всех обезличенных сделок (по всем инструментам)
   Период: 2022-01-01 — 2022-01-01
   Папка: D:\sd2\tinvest_data\2022




Скачивание дней:   0%|                                                                           | 0/1 [00:00<?, ?it/s]

... (17 outputs left out)


Скачивание дней:   0%|                                                                           | 0/1 [00:00<?, ?it/s]

⚠️  2022-01-09 — файл слишком маленький (84 байт)


Докачка:   3%|██                                                                       | 3/108 [00:06<03:58,  2.27s/it]


KeyboardInterrupt: 

In [5]:



def download_tbank_all_trades_by_days(
    start_date: str,
    end_date: str = None,
    output_folder: str = "tbank_all_trades",
    delay: float = 0.85,
    max_retries: int = 5,
    min_file_size: int = 2000,
    overwrite: bool = False
):
    """
    Скачивает обезличенные сделки ЗА ВСЕ инструменты по дням с повышенной устойчивостью.
    Каждый день сохраняется в файл: YYYY-MM-DD.csv.gz
    """
    if end_date is None:
        end_date = (datetime.today() - pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    os.makedirs(output_folder, exist_ok=True)
    dates = pd.date_range(start=start_date, end=end_date, freq='D')

    print(f"🚀 Скачивание всех обезличенных сделок (по всем инструментам)")
    print(f"   Период: {start_date} — {end_date}")
    print(f"   Папка: {output_folder}\n")

    downloaded = 0
    skipped = 0
    small_files = 0
    failed_dates = []

    for date_obj in tqdm(dates, desc="Скачивание дней"):
        date_str = date_obj.strftime('%Y-%m-%d')
        url = f"https://invest-public-api.tinkoff.ru/history-trades/{date_str}"
        
        filename = f"{date_str}.csv.gz"
        full_path = os.path.join(output_folder, filename)

        # Пропускаем уже нормально скачанные файлы
        if not overwrite and os.path.exists(full_path) and os.path.getsize(full_path) >= min_file_size:
            skipped += 1
            continue

        success = False
        for attempt in range(1, max_retries + 1):
            try:
                # Ключевые изменения здесь:
                response = requests.get(url, timeout=90, stream=True)
                response.raise_for_status()

                # Записываем файл частями — это лучше всего решает InvalidChunkLength
                with open(full_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):  # 8 КБ
                        if chunk:
                            f.write(chunk)

                size_bytes = os.path.getsize(full_path)
                size_mb = size_bytes / (1024 * 1024)

                if size_bytes < min_file_size:
                    small_files += 1
                    print(f"⚠️  {date_str} — файл слишком маленький ({size_bytes} байт)")
                    if os.path.exists(full_path):
                        os.remove(full_path)
                    break

                print(f"✅ {date_str} — скачано {size_mb:.2f} МБ (попытка {attempt})")
                downloaded += 1
                success = True
                break

            except (ChunkedEncodingError, ConnectionError, Timeout, RequestException) as e:
                print(f"   ⚠️  {date_str} — {type(e).__name__} (попытка {attempt}/{max_retries})")
                if os.path.exists(full_path):
                    os.remove(full_path)   # удаляем повреждённый файл

                if attempt < max_retries:
                    sleep_time = (2 ** (attempt - 1)) + random.uniform(0.5, 2.0)
                    time.sleep(sleep_time)
                else:
                    failed_dates.append(date_str)

            except Exception as e:
                print(f"   ❌ {date_str} — неожиданная ошибка: {e}")
                if os.path.exists(full_path):
                    os.remove(full_path)
                failed_dates.append(date_str)
                break

        if not success:
            failed_dates.append(date_str)

        # Пауза между днями
        time.sleep(delay + random.uniform(0.1, 0.5))

    # Итоговая статистика
    print("\n" + "="*80)
    print("🎉 Скачивание завершено!")
    print(f"   Скачано новых файлов : {downloaded}")
    print(f"   Пропущено (уже есть) : {skipped}")
    print(f"   Слишком маленьких    : {small_files}")
    print(f"   Не удалось скачать   : {len(failed_dates)}")
    print(f"   Всего обработано дней: {len(dates)}")
    print(f"   Папка: {os.path.abspath(output_folder)}")

    if failed_dates:
        print("\n❌ Даты, которые не удалось скачать:")
        for d in failed_dates[:15]:
            print(f"   • {d}")
        if len(failed_dates) > 15:
            print(f"   ... и ещё {len(failed_dates)-15} дат")

    return failed_dates



In [ ]:
import pandas as pd
import gzip
import os
from pathlib import Path
from tqdm import tqdm
from typing import List, Union

def extract_trades_by_tickers(
    folder_path: str,
    tickers: Union[str, List[str]],
    output_folder: str = "extracted_trades",
    save_separately: bool = True,
    verbose: bool = True
):
    """
    Извлекает обезличенные сделки только по указанным тикерам из архивов .csv.gz
    
    Параметры:
        folder_path      : путь к папке с файлами YYYY-MM-DD.csv.gz
        tickers          : один тикер (str) или список тикеров ['SBER', 'GAZP']
        output_folder    : куда сохранять результат
        save_separately  : True = сохранить отдельный файл для каждого тикера
                          False = сохранить всё в один большой файл
        verbose          : выводить прогресс и информацию
    """
    
    if isinstance(tickers, str):
        tickers = [tickers]
    
    tickers = [t.upper() for t in tickers]  # приводим к верхнему регистру
    
    os.makedirs(output_folder, exist_ok=True)
    
    # Находим все .csv.gz файлы
    gz_files = sorted([f for f in os.listdir(folder_path) 
                      if f.endswith('.csv.gz')])
    
    if not gz_files:
        raise FileNotFoundError(f"В папке {folder_path} не найдено файлов .csv.gz")
    
    print(f"Найдено {len(gz_files)} архивных файлов")
    print(f"Ищем сделки по тикерам: {tickers}\n")
    
    # Словарь для хранения данных по каждому тикеру
    data_by_ticker = {ticker: [] for ticker in tickers}
    
    for filename in tqdm(gz_files, desc="Обработка дней"):
        full_path = os.path.join(folder_path, filename)
        date_str = filename.replace('.csv.gz', '')
        
        try:
            # Читаем gz-файл напрямую
            with gzip.open(full_path, 'rt', encoding='utf-8') as f:
                df = pd.read_csv(f, low_memory=False)
            
            # Предполагаем, что колонка с тикером называется 'secid', 'ticker' или 'symbol'
            # Подстраиваемся под реальные названия колонок
            ticker_col = None
            for col in ['secid', 'ticker', 'symbol', 'instrument', 'TICKER']:
                if col in df.columns:
                    ticker_col = col
                    break
            
            if ticker_col is None:
                if verbose:
                    print(f"⚠️  В файле {filename} не найдена колонка с тикером")
                continue
            
            # Фильтруем только нужные тикеры
            mask = df[ticker_col].astype(str).str.upper().isin(tickers)
            df_filtered = df[mask].copy()
            
            if not df_filtered.empty:
                df_filtered['date'] = date_str  # добавляем дату для удобства
                for ticker in tickers:
                    ticker_data = df_filtered[df_filtered[ticker_col].astype(str).str.upper() == ticker]
                    if not ticker_data.empty:
                        data_by_ticker[ticker].append(ticker_data)
                        
        except Exception as e:
            print(f"❌ Ошибка при обработке {filename}: {e}")
            continue
    
    # Объединяем данные и сохраняем
    total_saved = 0
    
    for ticker in tickers:
        if not data_by_ticker[ticker]:
            print(f"⚠️  По тикеру {ticker} данных не найдено")
            continue
            
        result_df = pd.concat(data_by_ticker[ticker], ignore_index=True)
        result_df = result_df.sort_values(by=['date', 'trade_time' if 'trade_time' in result_df.columns else 'begin'])
        
        if save_separately:
            output_file = os.path.join(output_folder, f"{ticker}_trades.csv")
            result_df.to_csv(output_file, index=False)
            print(f"✅ {ticker}: сохранено {len(result_df):,} сделок → {output_file}")
        else:
            # Добавляем колонку с тикером, если сохраняем всё вместе
            result_df['ticker'] = ticker
        
        total_saved += len(result_df)
    
    # Если сохраняем всё в один файл
    if not save_separately and total_saved > 0:
        all_df = pd.concat([pd.concat(data_by_ticker[t]) for t in tickers if data_by_ticker[t]], 
                          ignore_index=True)
        output_file = os.path.join(output_folder, "all_selected_trades.csv")
        all_df.to_csv(output_file, index=False)
        print(f"✅ Все тикеры сохранены в один файл: {output_file} ({total_saved:,} сделок)")
    
    print(f"\n🎉 Извлечение завершено! Всего сохранено сделок: {total_saved:,}")
    return data_by_ticker




In [ ]:
# ====================== ПРИМЕР ИСПОЛЬЗОВАНИЯ ======================

if __name__ == "__main__":
    # Один тикер
    extract_trades_by_tickers(
        folder_path="tbank_all_trades",
        tickers="SBER",
        output_folder="extracted_SBER",
        save_separately=True
    )

    # Несколько тикеров
    # extract_trades_by_tickers(
    #     folder_path="tbank_all_trades",
    #     tickers=["SBER", "GAZP", "LKOH", "VTBR"],
    #     output_folder="extracted_liquid",
    #     save_separately=True
    # )